# LLM vs. LLM: DistilBERT baseline on Colab (step 06, TASK_06 section 3)

Trains `distilbert-base-uncased` once on the `train` partition (traditional phishing + Enron ham, the Nahmias
protocol) with the fixed settings of `configs/baselines.yaml` (3 epochs, lr 2e-5, batch 16, max 512 tokens,
seed from `configs/data.yaml`, fp32, deterministic kernels, no early stopping), scores `valid` (sanity
precision / recall / F1, recorded in `reports/baselines_report.json`) and then the 5,207 evaluation items of
the detectors, one row per item in the detector runner's schema, with `scripts/06_run_baselines.py`. The
checkpoint's sha256 is the `model_hash` of every row; the checkpoint itself (about 268 MB) is copied to Drive,
not to git. The two TF-IDF baselines run on the laptop and are not part of this notebook.

**Drive layout expected under `DRIVE_DIR`** (the same folder as `cloud_run.ipynb`):

```
llm-thesis/
  data/processed/records.jsonl, partitions.json, variants_eval.jsonl, variants_dev.jsonl   <- from the laptop
  results/raw/baseline_distilbert/supervised/{valid,eval}.jsonl   <- written here, synced after every cell
  reports/baselines_report.json                                    <- merged report (TF-IDF entries from the laptop are kept)
  models/baselines/distilbert.pt                                   <- the checkpoint (restored before scoring, so a
                                                                      disconnect after training does not retrain)
  logs/baseline_distilbert_{valid,eval}.log
```

Before running: Runtime > Change runtime type > T4 GPU. Run top to bottom. After a disconnect, run the whole
notebook again: the checkpoint and the partial results are restored from Drive and the eval cell resumes at
the first missing item. When finished, copy `results/raw/baseline_distilbert/`, `reports/baselines_report.json`
and `models/baselines/distilbert.pt` (kept outside git) to the laptop, run `python scripts/06_check_integrity.py`
and commit the results and the report.

In [ ]:
#@title 1. Parameters
REF = "main"                        #@param {type:"string"}
REPO_URL = "https://github.com/aliholmes5544/LLM.git"  #@param {type:"string"}
DRIVE_DIR = "/content/drive/MyDrive/llm-thesis"  #@param {type:"string"}
BASELINE = "distilbert"             #@param {type:"string"}
TIER = "colab_t4"
REPO_DIR = "/content/llm"
# REF: a tag (preferred, e.g. v0.6.4-baselines) or a branch; the commit actually used is recorded in the report.

In [ ]:
#@title 2. Mount Drive, check the GPU and the Drive layout
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
assert "T4" in gpu, f"tier {TIER} expects a T4 GPU, this runtime has {gpu!r} (Runtime > Change runtime type)"
print("GPU ok:", gpu, "| python", sys.version.split()[0])
REQUIRED = ["data/processed/records.jsonl", "data/processed/partitions.json",
            "data/processed/variants_eval.jsonl", "data/processed/variants_dev.jsonl"]
missing = [p for p in REQUIRED if not os.path.exists(os.path.join(DRIVE_DIR, p))]
assert not missing, f"missing on Drive under {DRIVE_DIR}: {missing}"
for d in ("results/raw", "reports", "logs", "models/baselines"):
    os.makedirs(os.path.join(DRIVE_DIR, d), exist_ok=True)

In [ ]:
#@title 3. Clone the repository at REF
import os, subprocess
os.chdir("/content")
if os.path.exists(REPO_DIR):
    subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(["git", "clone", "--quiet", "--branch", REF, "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
desc = subprocess.run(["git", "describe", "--tags", "--always"], capture_output=True, text=True).stdout.strip()
print("repository at", REPO_DIR, "commit", commit, "describe", desc)

In [ ]:
#@title 4. Install the pinned baseline requirements (torch: the Colab CUDA build, version recorded in the report)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-baselines.txt"], check=True)
import sklearn, torch, transformers, yaml
print("scikit-learn", sklearn.__version__, "| torch", torch.__version__, "cuda", torch.cuda.is_available(),
      "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "no CUDA device"

In [ ]:
#@title 5. Pull the processed data, restore the checkpoint, previous results and the report from Drive
import hashlib, json, os, shutil, subprocess, time
os.makedirs("data/processed", exist_ok=True)
for p in REQUIRED:
    shutil.copy2(os.path.join(DRIVE_DIR, p), p)
fixed = json.load(open("reports/variants_eval_fixed_artifacts.json"))["files"]["data/processed/variants_eval.jsonl"]
h = hashlib.sha256(open("data/processed/variants_eval.jsonl", "rb").read()).hexdigest()
assert h == fixed["sha256"], f"variants_eval.jsonl sha256 {h} differs from the fixed artefact {fixed['sha256']}"
print("variants_eval.jsonl verified")

RESULT_DIR = f"results/raw/baseline_{BASELINE}"
CKPT = f"models/baselines/{BASELINE}.pt"
REPORT = "reports/baselines_report.json"

def sync_to_drive():
    if os.path.isdir(RESULT_DIR):
        os.makedirs(os.path.join(DRIVE_DIR, RESULT_DIR), exist_ok=True)
        subprocess.run(["rsync", "-a", RESULT_DIR + "/", os.path.join(DRIVE_DIR, RESULT_DIR) + "/"], check=True)
    for f in (REPORT, CKPT):
        dst = os.path.join(DRIVE_DIR, f)
        same_size = os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(f) if os.path.exists(f) else True
        if os.path.exists(f) and (f == REPORT or not same_size):     # the 268 MB checkpoint is copied once
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(f, dst)
    print(time.strftime("%H:%M:%S"), "synced to", DRIVE_DIR)

# the report in git holds the laptop's TF-IDF entries; the Drive copy, if newer, holds this baseline's entry too
for f in (REPORT, CKPT):
    src = os.path.join(DRIVE_DIR, f)
    if os.path.exists(src):
        os.makedirs(os.path.dirname(f), exist_ok=True)
        shutil.copy2(src, f)
        print("restored", f)
if os.path.isdir(os.path.join(DRIVE_DIR, RESULT_DIR)):
    os.makedirs(RESULT_DIR, exist_ok=True)
    subprocess.run(["rsync", "-a", os.path.join(DRIVE_DIR, RESULT_DIR) + "/", RESULT_DIR + "/"], check=True)
    print("restored", RESULT_DIR)

In [ ]:
#@title 6. Train (once) and score valid, then eval; synced to Drive after each cell
import os, subprocess, sys
for partition in ("valid", "eval"):
    log_path = os.path.join(DRIVE_DIR, "logs", f"baseline_{BASELINE}_{partition}.log")
    with open(log_path, "a") as lf:
        p = subprocess.Popen([sys.executable, "-u", "scripts/06_run_baselines.py", "--baseline", BASELINE,
                              "--partition", partition, "--tier", TIER, "--backend", "cuda"],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end="", flush=True)
            lf.write(line)
        rc = p.wait()
    sync_to_drive()
    if rc != 0:
        raise SystemExit(f"{BASELINE}/{partition} exited with code {rc}; see {log_path}; fix and run the notebook again (it resumes)")

In [ ]:
#@title 7. Summary
import json
rep = json.load(open("reports/baselines_report.json"))
b = rep["baselines"][BASELINE]
print("model_hash (checkpoint sha256):", b["model_hash"])
print("pretrained:", b.get("pretrained"), "| versions:", b["versions"])
print("training:", {k: v for k, v in b["training"].items()})
for part, c in b["cells"].items():
    print(part, {k: c[k] for k in ("n_done", "n_expected", "complete", "latency_median_s", "truncated_input", "pred_counts")},
          c.get("valid_metrics", ""))
print(f"Next: copy {DRIVE_DIR}/results/raw/baseline_{BASELINE}, {DRIVE_DIR}/reports/baselines_report.json and "
      f"{DRIVE_DIR}/models/baselines/{BASELINE}.pt to the laptop; run scripts/06_check_integrity.py; commit results and report.")